# Cython: From Python to Near-C Speed

This notebook is a **from-scratch to intermediate/advanced** guide to **Cython**: writing
Python-like code that compiles to C and runs much faster for CPU-bound hot paths.

We cover:

1. **What Cython is** and when it helps
2. **Basic syntax**: type annotations, `cdef`, `.pyx` files
3. **Building extensions**: `setup.py` / `pyproject.toml`, in-place build
4. **Intermediate**: `cdef` classes, `nogil`, and NumPy integration
5. **Study case**: a simple WebSocket data-streaming class — first in pure Python,
   then with a Cython-accelerated processing core; we measure and store per-message
   delays for comparison
6. **Cython vs Rust vs C++** for trading engines: latency, throughput, and trade-offs

> Cython is most useful for **CPU-bound** code (numeric loops, parsing, validation).
> I/O-bound code (network, disk) is rarely sped up much; we still show a full pipeline
> so you see how to integrate Cython into a streaming component.

### Contents

- [1. What Cython Is and When It Helps](#1-what-cython-is-and-when-it-helps)
- [2. Basic Syntax: Types and cdef](#2-basic-syntax-types-and-cdef)
- [3. Building from the Notebook](#3-building-from-the-notebook)
- [4. Intermediate: cdef Classes and nogil](#4-intermediate-cdef-classes-and-nogil)
- [5. Study Case: Simple Data-Streaming Class (Python vs Cython)](#5-study-case-simple-data-streaming-class-python-vs-cython)
- [6. Cython vs Rust vs C++ for Trading-Engine Components](#6-cython-vs-rust-vs-c-for-trading-engine-components)
- [7. Best Practices and Pitfalls](#7-best-practices-and-pitfalls)

## 1. What Cython Is and When It Helps

- **Cython** is a superset of Python that compiles to C and then to a native extension
  (`.pyd` / `.so`) loadable by Python.
- You add **type information** (e.g. `cdef int i`, `cdef double x`) so the compiler
  can generate fast C code instead of interpreted Python.
- Best for: **tight loops**, **numeric code**, **parsing/validation** of high-volume data.
- Less useful for: mostly I/O (network, file), or code that already delegates to
  fast C libraries (NumPy, etc.).

We'll write `.pyx` modules and build them; then use them from the notebook or from
Python scripts.

In [1]:
# 1.1 Check Cython availability

try:
    import Cython
    print("Cython version:", Cython.__version__)
except ImportError:
    print("Install Cython: pip install cython")

Cython version: 3.1.2


## 2. Basic Syntax: Types and `cdef`

In a `.pyx` file (or a cell with Cython magic), you can:

- Declare **local variables** with `cdef type name` for C-level speed.
- Annotate **function parameters and return types**.
- Use `def` for Python-callable functions and `cdef` for C-only (faster, not callable from Python).

Example: a tight loop that sums squares. Pure Python uses dynamic dispatch every iteration;
Cython with typed variables compiles to a simple C loop.

In [2]:
# 2.1 Pure Python version (baseline)

def sum_squares_py(n: int) -> int:
    total = 0
    for i in range(n):
        total += i * i
    return total


# Quick sanity check and timing
import time
t0 = time.perf_counter()
result = sum_squares_py(1_000_000)
elapsed = time.perf_counter() - t0
print(f"sum_squares_py(1_000_000) = {result}, took {elapsed*1000:.2f} ms")

sum_squares_py(1_000_000) = 333332833333500000, took 31.36 ms


### 2.2 Cython version (typed)

We'll create a `.pyx` file with the same logic but C-level types. The key additions:
- `cdef long i, total` for loop variables and accumulator
- Optional `cpdef` for a function callable from both Python and C (with a fast C path)

In [3]:
# 2.2 Write and build a minimal Cython module

# Write the .pyx source to the same directory as this notebook
CYTHON_SUM_SQUARES = """
# distutils: language=c
# cython: language_level=3

def sum_squares_cy(int n):
    cdef long i, total
    total = 0
    for i in range(n):
        total += i * i
    return total
"""

from pathlib import Path
# Build under cwd/cython_build (run notebook from training/ so files stay local)
_notebook_dir = Path(".").resolve()
_cython_dir = _notebook_dir / "cython_build"
_cython_dir.mkdir(exist_ok=True)
(_cython_dir / "sum_squares.pyx").write_text(CYTHON_SUM_SQUARES)
print("Wrote sum_squares.pyx to", _cython_dir)

Wrote sum_squares.pyx to C:\Users\GSL\Projects\Challenges\training\cython_build


In [4]:
# 2.3 setup.py for building the Cython extension

SETUP_PY = """
from setuptools import setup, Extension
from Cython.Build import cythonize

setup(
    ext_modules=cythonize(
        [Extension("sum_squares", ["sum_squares.pyx"])],
        compiler_directives={"language_level": "3"},
    )
)
"""
(_cython_dir / "setup.py").write_text(SETUP_PY)
print("Wrote setup.py")

Wrote setup.py


In [5]:
# 2.4 Build the extension and compare timings

import subprocess
import sys

result = subprocess.run(
    [sys.executable, "setup.py", "build_ext", "--inplace"],
    cwd=str(_cython_dir),
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    print("STDOUT:", result.stdout)
    print("STDERR:", result.stderr)
else:
    print("Build succeeded.")

# Import from the build directory
sys.path.insert(0, str(_cython_dir))
import sum_squares  # type: ignore

# Time comparison
n = 2_000_000
for name, func in [("Python", sum_squares_py), ("Cython", sum_squares.sum_squares_cy)]:
    t0 = time.perf_counter()
    r = func(n)
    elapsed = time.perf_counter() - t0
    print(f"{name}: result={r}, time={elapsed*1000:.2f} ms")

STDOUT: Compiling sum_squares.pyx because it changed.
[1/1] Cythonizing sum_squares.pyx
running build_ext
building 'sum_squares' extension

STDERR: error: Microsoft Visual C++ 14.0 or greater is required. Get it with "Microsoft C++ Build Tools": https://visualstudio.microsoft.com/visual-cpp-build-tools/



ModuleNotFoundError: No module named 'sum_squares'

## 3. Building from the Notebook

The previous cells wrote `sum_squares.pyx` and `setup.py` into a `cython_build` folder under
your current working directory, then ran `build_ext --inplace`. The compiled module
(`sum_squares*.pyd` or `sum_squares*.so`) is loaded with `import sum_squares`.

If the build fails, ensure you have a C compiler (e.g. Visual Studio Build Tools on Windows,
or `gcc` on Linux/macOS) and run the notebook from the directory that contains `cython.ipynb`,
or adjust `_notebook_dir` / `_cython_dir` to match your layout.

## 4. Intermediate: `cdef` Classes and `nogil`

- **`cdef class`**: extension types with C-level attribute access. Attributes are declared
  with `cdef`; no Python dict for instance state. Much faster than a normal class for
  hot-path data structures.
- **`nogil`**: inside a `with nogil:` block (or a `cdef ... nogil` function), you release
  the GIL so C code can run in parallel with other threads. Use for pure C-style loops
  that don't touch Python objects.

We'll add a small **message parser** in Cython that will be used in the streaming study
case: parse a simple numeric message (e.g. "price,size") into two doubles. This simulates
a hot path in a tick processor.

In [ ]:
# 4.1 Fast message parser in Cython (for use in study case)

# We'll create a small module that parses "price,size" lines and returns (price, size).
# This will be used by the streaming class to simulate a hot path.

PARSER_PYX = '''
# cython: language_level=3

def parse_price_size(str s):
    """Parse a line "price,size" into (float, float). Cython-optimized."""
    cdef list parts = s.split(",")
    if len(parts) != 2:
        return None
    try:
        return (float(parts[0]), float(parts[1]))
    except ValueError:
        return None
'''

# Write to same cython_build dir; we'll add it to the setup in a combined build below
(_cython_dir / "msg_parser.pyx").write_text(PARSER_PYX)
print("Wrote msg_parser.pyx")

In [ ]:
# 4.2 Build msg_parser extension

SETUP_PARSER = """
from setuptools import setup, Extension
from Cython.Build import cythonize
setup(ext_modules=cythonize([Extension("msg_parser", ["msg_parser.pyx"])], language_level="3"))
"""
(_cython_dir / "setup_parser.py").write_text(SETUP_PARSER)
subprocess.run([sys.executable, "setup_parser.py", "build_ext", "--inplace"], cwd=str(_cython_dir), capture_output=True)
if "msg_parser" in sys.modules:
    del sys.modules["msg_parser"]
sys.path.insert(0, str(_cython_dir))
import msg_parser  # type: ignore  # loads compiled .pyd/.so from same dir
print("parse_price_size('100.5,10'):", msg_parser.parse_price_size("100.5,10"))
print("parse_price_size('bad'):", msg_parser.parse_price_size("bad"))

## 5. Study Case: Simple Data-Streaming Class (Python vs Cython)

We implement a **minimal data-streaming pipeline** that:

- Consumes a stream of text messages (simulating WebSocket or TCP frames).
- Parses each message (e.g. `"price,size"`) and runs a small processing step.
- Records **per-message processing delay** (time from message received to processing done).

First version: **pure Python** parser and loop. Second version: **Cython** parser for the
hot path. We then **store delays** (e.g. in lists and optionally to JSON) and compare.

To keep the example runnable without a live WebSocket server, the "stream" is a list of
messages; in production you would replace this with `async for msg in websocket:` or
a similar source.

In [ ]:
# 5.1 Pure Python parser and streamer

import time
import json
from pathlib import Path

def parse_price_size_py(s: str) -> tuple[float, float] | None:
    """Parse 'price,size' line in pure Python."""
    parts = s.strip().split(",")
    if len(parts) != 2:
        return None
    try:
        return (float(parts[0]), float(parts[1]))
    except ValueError:
        return None


class SimpleStreamerPy:
    """Process a stream of 'price,size' messages; record per-message delay (Python hot path)."""

    def __init__(self):
        self.delays_ms: list[float] = []
        self.processed: list[tuple[float, float]] = []

    def process_message(self, msg: str) -> None:
        t0 = time.perf_counter()
        parsed = parse_price_size_py(msg)  # pure Python parse
        if parsed is not None:
            self.processed.append(parsed)
        elapsed_ms = (time.perf_counter() - t0) * 1000
        self.delays_ms.append(elapsed_ms)  # store for comparison

    def run(self, messages: list[str]) -> None:
        self.delays_ms.clear()
        self.processed.clear()
        for msg in messages:
            self.process_message(msg)

    def save_delays(self, path: str | Path) -> None:
        with open(path, "w") as f:
            json.dump({"delays_ms": self.delays_ms, "mean_ms": sum(self.delays_ms) / len(self.delays_ms) if self.delays_ms else 0}, f, indent=2)

In [ ]:
# 5.2 Cython-accelerated streamer (same API, Cython parser)

class SimpleStreamerCy:
    """Same as SimpleStreamerPy but uses Cython msg_parser for the parse hot path."""

    def __init__(self):
        self.delays_ms: list[float] = []
        self.processed: list[tuple[float, float]] = []

    def process_message(self, msg: str) -> None:
        t0 = time.perf_counter()
        parsed = msg_parser.parse_price_size(msg)  # Cython implementation
        if parsed is not None:
            self.processed.append(parsed)
        elapsed_ms = (time.perf_counter() - t0) * 1000
        self.delays_ms.append(elapsed_ms)

    def run(self, messages: list[str]) -> None:
        self.delays_ms.clear()
        self.processed.clear()
        for msg in messages:
            self.process_message(msg)

    def save_delays(self, path: str | Path) -> None:
        with open(path, "w") as f:
            json.dump({"delays_ms": self.delays_ms, "mean_ms": sum(self.delays_ms) / len(self.delays_ms) if self.delays_ms else 0}, f, indent=2)

In [ ]:
# 5.3 Run both streamers, measure and store delays

# Simulate a stream of tick-like messages (e.g. from WebSocket)
MESSAGE_COUNT = 50_000
messages = [f"{100.0 + i * 0.01},{10 + i % 100}" for i in range(MESSAGE_COUNT)]

streamer_py = SimpleStreamerPy()
streamer_cy = SimpleStreamerCy()

# Python version
t0_py = time.perf_counter()
streamer_py.run(messages)
total_py = time.perf_counter() - t0_py
mean_delay_py = sum(streamer_py.delays_ms) / len(streamer_py.delays_ms) if streamer_py.delays_ms else 0

# Cython version
t0_cy = time.perf_counter()
streamer_cy.run(messages)
total_cy = time.perf_counter() - t0_cy
mean_delay_cy = sum(streamer_cy.delays_ms) / len(streamer_cy.delays_ms) if streamer_cy.delays_ms else 0

print(f"Messages: {MESSAGE_COUNT}")
print(f"Python:  total={total_py:.3f}s, mean delay per msg={mean_delay_py:.4f} ms")
print(f"Cython:  total={total_cy:.3f}s, mean delay per msg={mean_delay_cy:.4f} ms")
print(f"Speedup (total time): {total_py / total_cy:.2f}x")

# Store delays for later comparison (e.g. plot or analyze)
out_dir = Path(_cython_dir)
streamer_py.save_delays(out_dir / "delays_python.json")
streamer_cy.save_delays(out_dir / "delays_cython.json")
print("Saved delays to", out_dir / "delays_python.json", "and delays_cython.json")

## 6. Cython vs Rust vs C++ for Trading-Engine Components

Below is an **approximate, realistic** comparison for hot-path components (order handling,
tick parsing, risk checks, book building) in terms of latency, throughput, and
operational trade-offs. Numbers are order-of-magnitude and depend heavily on
implementation quality and workload.

| Dimension | Pure Python | Cython (typed hot path) | Rust | C++ |
|-----------|-------------|--------------------------|------|-----|
| **Typical hot-path latency** | 10–100+ µs | 1–10 µs | 0.5–5 µs | 0.5–5 µs |
| **Worst-case / tail latency** | 100 µs – ms | 10–50 µs | 5–20 µs | 5–20 µs |
| **Throughput (ops/sec, single core)** | 10⁴–10⁵ | 10⁵–10⁶ | 10⁶–10⁷ | 10⁶–10⁷ |
| **Memory overhead** | High (objects, indirection) | Medium (C structs in hot path) | Low | Low |
| **Development speed** | Fast | Medium (incremental from Python) | Slower (new language, borrow checker) | Slower (manual memory, build) |
| **Ecosystem / libraries** | Very rich | Python + C libs | Growing (async, serialization) | Mature (low-level libs) |
| **Safety (memory, concurrency)** | Safe | Safe (GIL by default) | Safe (ownership) | Manual (easy to get wrong) |
| **Deployment** | Interpreted or .pyc | Compiled extension (.so/.pyd) | Single binary | Binary + libs |

**When to choose what**

- **Cython**: You already have Python trading/quant code and want to **speed up specific
  hot paths** (parsing, numeric loops, book updates) without rewriting the whole stack.
  Latency can get within a **small multiple** (roughly 2–5×) of hand-tuned C/Rust for
  similar algorithms. Integration is smooth (same process, same GIL/threading model).

- **Rust**: Good when you need **predictable low latency**, **no GIL**, and **memory safety**
  for a new service or core engine. Typical use: new order/tick gateway or risk service
  where you control the whole process. Latency and throughput are in the same ballpark
  as C++ with less undefined behavior.

- **C++**: Still common in **ultra-low-latency** and legacy engines. Maximum control and
  maturity of HFT libraries and toolchains; higher risk of bugs and harder refactoring
  than Rust or Cython.

**Summary**

For **trading engines**, Cython is a practical way to get **order-of-magnitude** latency
and throughput improvements over pure Python on CPU-bound hot paths, while staying
inside the Python ecosystem. Rust and C++ target **lower absolute latency** and
**higher throughput** when the whole component is implemented in a systems language;
the cost is development time and (for C++) safety. Approximate relative latency:
**Python : Cython : Rust/C++ ≈ 10 : 2 : 1** for a typical hot path (e.g. parse + validate
one message).

## 7. Best Practices and Pitfalls

- **Profile first**: Use `cProfile` or `py-spy` to find the real hot path before
  rewriting it in Cython.
- **Type the hot loop**: Use `cdef` for loop variables and accumulators; avoid
  Python object access inside tight loops.
- **Keep Python at the edges**: Let Cython handle the numeric/parsing core; keep
  APIs and I/O in Python for simplicity.
- **Build reproducibly**: Pin Cython and compiler versions in CI; use `pyproject.toml`
  or `setup.py` so others can build the extension.
- **Watch the GIL**: For parallelism, use `nogil` in C-only sections or offload
  to ProcessPoolExecutor; threads that only do Cython/nogil code can run in parallel.